In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("current_week", "auto", ["auto"] + [str(i) for i in range(1, 17)], "Current Week")
dbutils.widgets.dropdown("buybacks_used", "0", ["0", "1", "2"], "Buybacks Used")
dbutils.widgets.text(
    "used_teams_manual",
    "",
    "Teams Already Picked (comma-separated)"
)

In [0]:
used_teams_manual_raw = dbutils.widgets.get("used_teams_manual")

# Convert comma-separated text into a clean Python list
used_teams_manual = [
    t.strip() for t in used_teams_manual_raw.split(",") if t.strip() != ""
]

used_teams_manual

In [0]:
import pandas as pd

# Load original table (from your CSV import)
orig = spark.table("workspace.default.nfl_survivor_csv_export").toPandas()

# Clean column names to match predictions table
orig.columns = (
    orig.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Filter rows that already had a Target value
orig_filled = orig[orig["Target"].notna()].copy()

# Load predicted table
pred = spark.table("workspace.default.nfl_survivor_predictions").toPandas()

# Clean predicted column names too
pred.columns = (
    pred.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Replace Target with Predicted_Target for predicted rows
pred = pred.copy()
pred["Target"] = pred["Predicted_Target"]

# Drop Predicted_Target so columns match exactly
pred = pred.drop(columns=["Predicted_Target"])

# Combine original filled rows + predicted rows
combined = pd.concat([orig_filled, pred], ignore_index=True)

# Sort by Week for readability
combined = combined.sort_values("Week")

combined

# Coerce mixed-type columns to string for consistent Arrow conversion
# (orig table has comma-formatted strings, pred table has numerics)
for col in combined.columns:
    if combined[col].dtype == 'object':
        combined[col] = combined[col].astype(str)

combined_spark = spark.createDataFrame(combined)

combined_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.nfl_survivor_full_targets"
)

In [0]:
# Load merged table (original + predicted)
df = spark.table("workspace.default.nfl_survivor_full_targets").toPandas()

# Clean column names
df.columns = (
    df.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

# Create a unified risk column
# If Predicted_Target exists, use it
# Otherwise use Target
df["RiskScore"] = df["Predicted_Target"] if "Predicted_Target" in df.columns else df["Target"]

# If both exist (merged table), prefer Predicted_Target when available
if "Predicted_Target" in df.columns:
    df["RiskScore"] = df["Predicted_Target"].fillna(df["Target"])


In [0]:
current_week_raw = dbutils.widgets.get("current_week").strip().lower()
if current_week_raw in ("", "auto"):
    # Set by the Ingest Odds task in the job; when run by hand, ask ESPN directly
    try:
        current_week = int(dbutils.jobs.taskValues.get(taskKey="ingest_odds", key="current_week"))
    except Exception:
        current_week = None
    if current_week is None:
        try:
            # Written by fetch_nfl_data.py when the workspace has no internet
            with open("/Volumes/workspace/default/nfl_survivor/current_week.txt") as f:
                current_week = int(f.read().strip())
        except Exception:
            current_week = None
    if current_week is None:
        import requests
        board = requests.get("https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard", timeout=20).json()
        current_week = int(board.get("week", {}).get("number", 1))
        if board.get("season", {}).get("type") == 1:
            current_week = 1
else:
    current_week = int(current_week_raw)
print(f"Using current_week = {current_week}")
buybacks_used = int(dbutils.widgets.get("buybacks_used"))

df = combined.copy()
# Make sure odds sort as numbers, not text
df["Target"] = pd.to_numeric(df["Target"], errors="coerce")
df["Week"] = pd.to_numeric(df["Week"], errors="coerce").astype(int)


In [0]:
df.columns = (
    df.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('[,;{}()\n\t=]', '_', regex=True)
)

In [0]:
late = df[(df["Week"] >= 10) & (df["Week"] <= 16)].copy()
early = df[(df["Week"] <= 9) & (df["Week"] >= current_week)].copy()

In [0]:
import itertools
import pandas as pd
import numpy as np

# Convert moneyline Target into win probability
# Moneyline → implied probability
def moneyline_to_prob(ml):
    ml = float(ml)
    if ml < 0:
        return (-ml) / ((-ml) + 100)
    else:
        return 100 / (ml + 100)

# Build late-season probability table
late_probs = late.copy()
late_probs["Prob"] = late_probs["Target"].astype(float).apply(moneyline_to_prob)

# All unique teams appearing in late season
late_teams = sorted(late_probs["Team"].unique())

pair_results = []

# Evaluate every 2-team combination
for teamA, teamB in itertools.combinations(late_teams, 2):

    week_probs = []

    for week in range(10, 17):
        w = late_probs[late_probs["Week"] == week]

        # Get rows for each team
        rowA = w[w["Team"] == teamA]
        rowB = w[w["Team"] == teamB]

        # If a team doesn't play that week, treat probability as 0
        probA = rowA["Prob"].iloc[0] if len(rowA) > 0 else 0
        probB = rowB["Prob"].iloc[0] if len(rowB) > 0 else 0

        # Best of the two teams for that week
        best_prob = max(probA, probB)
        week_probs.append(best_prob)

    # Product of best weekly probabilities
    total_product = np.prod(week_probs)

    pair_results.append({
        "TeamA": teamA,
        "TeamB": teamB,
        "WeeklyProbs": week_probs,
        "TotalProduct": total_product
    })

# Rank pairs by total product
pairs_df = pd.DataFrame(pair_results).sort_values("TotalProduct", ascending=False)

pairs_df


In [0]:
late_picks = []
used_late_teams = set()

for week in range(10, 17):
    week_df = late[late["Week"] == week].sort_values("Target", ascending=True)

    for _, row in week_df.iterrows():
        team = row["Team"]
        opp = row["Team_Opp"]
        score = row["Target"]

        if team in used_teams_manual:
            continue

        if team not in used_late_teams:
            late_picks.append((week, team, opp, score))
            used_late_teams.add(team)
            break

In [0]:
import pandas as pd

late_df = pd.DataFrame(late_picks, columns=["Week", "Team", "Team_Opp", "Score"])
reserved_teams = set(late_df["Team"])
late_df

In [0]:
used_teams = set()
buybacks_remaining = 2 - buybacks_used

early_picks = []

for week in range(current_week, 8):
    week_df = early[early["Week"] == week].sort_values("Target", ascending=True)

    for _, row in week_df.iterrows():
        team = row["Team"]
        opp = row["Team_Opp"]
        score = row["Target"]

        # Skip manually-used teams
        if team in used_teams_manual:
            continue

        # Skip teams reserved for late season
        if team in reserved_teams:
            continue

        # Skip teams already used in early season
        if team in used_teams:
            continue

        # Risky pick logic
        if score > 0 and buybacks_remaining > 0:
            early_picks.append((week, team, opp, score, "Risky (Buyback Used)"))
            used_teams.add(team)
            buybacks_remaining -= 1
            break

        # Safe pick
        early_picks.append((week, team, opp, score, "Safe"))
        used_teams.add(team)
        break


In [0]:
early_df = pd.DataFrame(early_picks, columns=["Week", "Team", "Team_Opp", "Score", "Type"])
early_df


In [0]:
late_df["Type"] = "Reserved Late"
final_df = pd.concat([early_df, late_df]).sort_values("Week")
final_df


In [0]:
import pandas as pd

# Optional widget for interactive team selection
dbutils.widgets.text("team_lookup", "", "Team to Inspect")
team_lookup = dbutils.widgets.get("team_lookup").strip()

# If you prefer hard-coding, uncomment:
# team_lookup = "DEN"

if team_lookup == "":
    print("Please enter a team abbreviation in the widget.")
else:
    # Convert moneyline to probability
    def moneyline_to_prob(ml):
        ml = float(ml)
        if ml < 0:
            return (-ml) / ((-ml) + 100)
        else:
            return 100 / (ml + 100)

    team_df = df[df["Team"] == team_lookup].copy()

    # Clean and compute probability
    team_df["Target"] = team_df["Target"].astype(float)
    team_df["WinProb"] = team_df["Target"].apply(moneyline_to_prob)

    # Sort by week
    team_df = team_df.sort_values("Week")

    # Select useful columns
    display(team_df[[
        "Week",
        "Team",
        "Team_Opp",
        "Home",
        "Target",
        "WinProb"
    ]])


In [0]:
final_spark = spark.createDataFrame(final_df)
final_spark.write.mode("overwrite").saveAsTable("workspace.default.nfl_survivor_optimal_picks")

# Show the plan in the job run output
display(final_df)
